# SC-Mamba: Multivariate Comprehensive Benchmark (17 Datasets)
## Hướng nghiên cứu 3 (Mandatory Benchmark)
Notebook này tự động lặp qua 17 datasets, tự động thiết lập cấu hình `N>1`, `pred_len`, và `context_len` tối ưu, huấn luyện từ đầu (Train from scratch) và trích xuất điểm MASE, CRPS, NLL tổng hợp thành bảng.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers==4.43.3 sentence-transformers torch==2.4.0 torchvision torchaudio ninja
!pip install numpy==1.26.4 pandas==2.2.2 scipy==1.13.1 scikit-learn==1.4.2 opt-einsum matplotlib
!pip install pyyaml tqdm einops wandb torchmetrics
!pip install matplotlib xgboost statsmodels properscoring utilsforecast kaleido plotly statsforecast
!pip install packaging

# Build flash-attention and causal-conv1d from source (requires ninja)
!pip install causal-conv1d>=1.4.0
!pip install mamba-ssm>=2.2.4 

## Setup Repository Workspace

In [ ]:
import os
project_path = '/content/drive/MyDrive/Colab Notebooks/SCMamba/SC-Mamba-Code'
os.chdir(project_path)
print("Current Working Directory:", os.getcwd())

## Comprehensive Benchmark Loop (Auto-Derivation)

In [ ]:
import os
import subprocess
import yaml
from pathlib import Path

# 1. Registry từ eval_real_dataset.py
REAL_DATASET_ASSETS = {
    "exchange_rate": 8, "cif_2016": 72, "nn5_daily": 111, "hospital": 767, 
    "covid_deaths": 266, "m1_yearly": 181, "m1_quarterly": 203, "m1_monthly": 617, 
    "m3_yearly": 645, "m3_quarterly": 756, "m3_monthly": 1428, "m3_other": 174, 
    "m4_yearly": 23000, "m4_quarterly": 24000, "m4_monthly": 48000, "m4_weekly": 359, 
    "m4_daily": 4227, "m4_hourly": 414
}
REAL_DATASETS = {
    "m1_yearly": 6, "m1_quarterly": 8, "m1_monthly": 18, 
    "m3_yearly": 6, "m3_quarterly": 8, "m3_monthly": 18, "m3_other": 8,
    "m4_yearly": 6, "m4_quarterly": 8, "m4_monthly": 18, "m4_weekly": 13, 
    "m4_daily": 14, "m4_hourly": 48, 
    "cif_2016": 12, "nn5_daily": 56, "hospital": 12, 
    "covid_deaths": 30, "exchange_rate": 30
}

# 17 Datasets (Bỏ qua m4 siêu lớn nếu VRAM Colab T4 không đủ)
COMPREHENSIVE_DATASETS = [
    "exchange_rate", "cif_2016", "nn5_daily", "hospital", "covid_deaths",
    "m1_yearly", "m1_quarterly", "m1_monthly",
    "m3_yearly", "m3_quarterly", "m3_monthly", "m3_other"
    # Bổ sung m4 tuỳ vào VRAM
]

def update_yaml(dataset):
    yaml_path = 'core/config.yaml'
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # ── Auto-Derivation (Smart Baseline) ──
    p_len = REAL_DATASETS[dataset]
    n_assets = REAL_DATASET_ASSETS[dataset]
    
    # Giới hạn N=800 để tránh OOM trên T4, code SC-Mamba sẽ tự động Sub-sample
    # (Hoặc để nguyên nếu kiến trúc chunk_size mới đã đủ gánh)
    max_vram_assets = min(n_assets, 767) 
    
    config['real_train_datasets'] = [dataset]
    config['pred_len'] = p_len
    config['context_len'] = p_len * 2 # Hoặc 256/512 nếu muốn test LRD
    config['num_assets'] = max_vram_assets
    
    # Cross-Asset Graph settings
    config['prior_config']['prior_mix_frac'] = 0.3 # 70% Real data focus
    config['beta_kl'] = 0.2
    config['num_epochs'] = 30
    config['batch_size'] = 4 if max_vram_assets > 100 else 8
    
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    return config['context_len'], max_vram_assets, p_len

benchmark_log = "logs/log_multivariate_benchmark.txt"

with open(benchmark_log, "a") as fout:
    fout.write("\n================ CROSS-ASSET BENCHMARK START ================\n")

for ds in COMPREHENSIVE_DATASETS:
    print(f"\n[BENCHMARK] Bắt đầu xử lý {ds}...")
    try:
        ctx, n_ass, p_len = update_yaml(ds)
        print(f"  → Configured: N={n_ass}, P_L={p_len}, Ctx={ctx}")
        
        # Chạy main.py
        result = subprocess.run(
            ["python", "core/main.py"], 
            capture_output=True, text=True, check=True
        )
        
        # Parse & Save Results
        log_file = f"logs/log_{ds}.txt"
        if os.path.exists(log_file):
            with open(log_file, "r") as f:
                lines = f.readlines()
            
            # Trích xuất 10 dòng cuối cùng (thường chứa table điểm)
            metrics = "".join(lines[-15:])
            
            with open(benchmark_log, "a") as fout:
                fout.write(f"\n--- {ds} (N={n_ass}) ---\n")
                fout.write(metrics)
            print(f"  ✅ Hoàn tất {ds}. Đã lưu vào {benchmark_log}")
        else:
            print(f"  ⚠️ Xong {ds} nhưng không tìm thấy {log_file}")
            
    except subprocess.CalledProcessError as e:
        print(f"  ❌ LÔI KHI CHẠY {ds}!")
        print("--- STDOUT ---")
        print(e.stdout[-1000:])
        print("--- STDERR ---")
        print(e.stderr[-1000:])
        
        with open(benchmark_log, "a") as fout:
            fout.write(f"\n--- {ds} FAILED ---\n")
        continue  # Bỏ qua và chạy dataset tiếp theo
    except Exception as ex:
        print(f"  ❌ UNKNOWN ERROR {ds}: {ex}")
        continue

print(f"\n🎉 BENCHMARK HOÀN TẤT! Xem kết quả mở tệp {benchmark_log}")
